# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided example for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via its Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
# Access the metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and associated Croissant `@id`s.

For each record set, we will print its `@id`, name, and its fields with their respective `@id`s and types.

In [ ]:
print('Record sets in this dataset:')

record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- Record set @id: {rs.id}\n  Name: {getattr(rs, 'name', '')}\n  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, Name: {getattr(field, 'name', '')}, Type: {getattr(field, 'data_type', '')}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

Here, we demonstrate extracting one record set into a DataFrame.

In [ ]:
# For demonstration, use the first record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")

# Show column names for the first record set
chosen_record_set_id = record_set_ids[0]  # Change index if you want a different one
print(f"Columns in '{chosen_record_set_id}':")
print(dataframes[chosen_record_set_id].columns.tolist())
dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

In this example, we select a numeric field and filter, normalize and group data accordingly.

In [ ]:
# Pick a numeric field to analyze (modify as appropriate per your dataset)
df = dataframes[chosen_record_set_id]
print('Numeric columns:')
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print(numeric_cols)

if len(numeric_cols) == 0:
    print("No numeric columns available for EDA in this record set.")
else:
    numeric_field = numeric_cols[0]  # choose first numeric column
    threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
    print(f"Normalized {numeric_field}:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field if available
    group_candidates = df.select_dtypes(include=['object']).columns.tolist()
    group_field = group_candidates[0] if len(group_candidates) > 0 else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if len(numeric_cols) > 0:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {numeric_field}')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field, by=group_field, rot=90)
        plt.ylabel(numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and perform basic analysis of a dataset defined by a Croissant schema using the `mlcroissant` library. We overviewed the record sets and fields, loaded one into a DataFrame, and performed simple filtering, normalization, grouping, and basic visualization. For more detailed analysis, refine field selection in EDA and customize the workflow based on the actual dataset content.